# OSM-Daten pro Bundesland herunterladen und filtern

Dieses Skript lädt OSM-PBF-Dateien für jedes deutsche Bundesland von Geofabrik herunter,
filtert alle Straßen (highways) mit Osmium,
und speichert die gefilterten PBF-Dateien.

**Vorteil:** Jedes Bundesland kann später einzeln geladen werden → sehr RAM-effizient!

**Output:** `processed_osm_files/processed_highways_DE-XX_latest.pbf` (~50-500 MB je nach Bundesland)

In [1]:
import os
import time
import random
import requests
import subprocess
import json
from datetime import datetime
from tqdm import tqdm

import geopandas as gpd
import pandas as pd

from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

from config import GEOFABRIK_CONFIG, PROCESSING_CONFIG, MAPILLARY_CONFIG, TILES_CONFIG

In [2]:
## Check if osmium is installed

# try:
#     result = subprocess.run(['osmium', '--version'], check=True, capture_output=True, text=True)
#     print(f"Osmium version: {result.stdout.strip()}")
# except subprocess.CalledProcessError as e:
#     print(f"Error running Osmium: {e}")

## Downloading OSM data from Geofabrik

In [3]:
# Geofabrik URLs für deutsche Bundesländer
# https://download.geofabrik.de/europe/germany.html

ALL_BUNDESLAND_URLS = {
    "DE-BW": "https://download.geofabrik.de/europe/germany/baden-wuerttemberg-latest.osm.pbf",
    "DE-BY": "https://download.geofabrik.de/europe/germany/bayern-latest.osm.pbf",
    "DE-BE": "https://download.geofabrik.de/europe/germany/berlin-latest.osm.pbf",
    "DE-BB": "https://download.geofabrik.de/europe/germany/brandenburg-latest.osm.pbf",
    "DE-HB": "https://download.geofabrik.de/europe/germany/bremen-latest.osm.pbf",
    "DE-HH": "https://download.geofabrik.de/europe/germany/hamburg-latest.osm.pbf",
    "DE-HE": "https://download.geofabrik.de/europe/germany/hessen-latest.osm.pbf",
    "DE-MV": "https://download.geofabrik.de/europe/germany/mecklenburg-vorpommern-latest.osm.pbf",
    "DE-NI": "https://download.geofabrik.de/europe/germany/niedersachsen-latest.osm.pbf",
    "DE-NW": "https://download.geofabrik.de/europe/germany/nordrhein-westfalen-latest.osm.pbf",
    "DE-RP": "https://download.geofabrik.de/europe/germany/rheinland-pfalz-latest.osm.pbf",
    "DE-SL": "https://download.geofabrik.de/europe/germany/saarland-latest.osm.pbf",
    "DE-SN": "https://download.geofabrik.de/europe/germany/sachsen-latest.osm.pbf",
    "DE-ST": "https://download.geofabrik.de/europe/germany/sachsen-anhalt-latest.osm.pbf",
    "DE-SH": "https://download.geofabrik.de/europe/germany/schleswig-holstein-latest.osm.pbf",
    "DE-TH": "https://download.geofabrik.de/europe/germany/thueringen-latest.osm.pbf",
}

# Filter Bundesländer basierend auf config.py
selected_bundeslaender = GEOFABRIK_CONFIG.get("bundeslaender")
if selected_bundeslaender:
    # Nur die ausgewählten Bundesländer verarbeiten
    BUNDESLAND_URLS = {k: v for k, v in ALL_BUNDESLAND_URLS.items() if k in selected_bundeslaender}
    print(f"🎯 Ausgewählte Bundesländer: {len(BUNDESLAND_URLS)} von {len(ALL_BUNDESLAND_URLS)}")
else:
    # Alle Bundesländer verarbeiten
    BUNDESLAND_URLS = ALL_BUNDESLAND_URLS
    print(f"📍 Alle Bundesländer: {len(BUNDESLAND_URLS)}")

print(f"{'─'*70}")
for bl, url in BUNDESLAND_URLS.items():
    print(f"  {bl}: {url.split('/')[-1]}")

📍 Alle Bundesländer: 16
──────────────────────────────────────────────────────────────────────
  DE-BW: baden-wuerttemberg-latest.osm.pbf
  DE-BY: bayern-latest.osm.pbf
  DE-BE: berlin-latest.osm.pbf
  DE-BB: brandenburg-latest.osm.pbf
  DE-HB: bremen-latest.osm.pbf
  DE-HH: hamburg-latest.osm.pbf
  DE-HE: hessen-latest.osm.pbf
  DE-MV: mecklenburg-vorpommern-latest.osm.pbf
  DE-NI: niedersachsen-latest.osm.pbf
  DE-NW: nordrhein-westfalen-latest.osm.pbf
  DE-RP: rheinland-pfalz-latest.osm.pbf
  DE-SL: saarland-latest.osm.pbf
  DE-SN: sachsen-latest.osm.pbf
  DE-ST: sachsen-anhalt-latest.osm.pbf
  DE-SH: schleswig-holstein-latest.osm.pbf
  DE-TH: thueringen-latest.osm.pbf


In [4]:
def should_download_osm_data(bundesland_code, max_age_days=None):
    """
    Prüft ob OSM-Daten für ein Bundesland heruntergeladen werden müssen.
    
    Berücksichtigt:
    - Existenz der gefilterten PBF-Datei
    - Alter der Datei (über osm_metadata.json)
    
    Args:
        bundesland_code: Der Bundesland-Code (z.B. "DE-BE")
        max_age_days: Maximales Alter in Tagen (default: aus config)
    
    Returns:
        True wenn Download nötig, False sonst
    """
    if max_age_days is None:
        max_age_days = PROCESSING_CONFIG.get('max_file_age_days', 4)
    
    # Prüfe ob gefilterte Datei existiert
    folder_processed = GEOFABRIK_CONFIG["processed_folder"]
    processed_file = os.path.join(
        folder_processed,
        f"processed_highways_{bundesland_code}_latest.pbf"
    )
    
    if not os.path.exists(processed_file):
        print(f"  ℹ️  Keine Datei vorhanden für {bundesland_code} → Download nötig")
        return True
    
    # Prüfe Alter über Metadata
    if not os.path.exists("output/osm_metadata.json"):
        print(f"  ℹ️  Keine Metadata vorhanden → Download nötig")
        return True
    
    try:
        with open("output/osm_metadata.json", "r") as f:
            metadata = json.load(f)
        
        bundeslaender = metadata.get("bundeslaender", {})
        if bundesland_code not in bundeslaender:
            print(f"  ℹ️  {bundesland_code} nicht in Metadata → Download nötig")
            return True
        
        # Parse processed_date (wann wurde die Datei verarbeitet)
        processed_date_str = metadata.get("processed_date")
        if not processed_date_str:
            print(f"  ℹ️  Kein processed_date in Metadata → Download nötig")
            return True
        
        processed_date = datetime.fromisoformat(processed_date_str.replace('Z', '+00:00'))
        
        # Berechne Alter seit Verarbeitung
        now = datetime.now(processed_date.tzinfo)
        age_days = (now - processed_date).days
        
        if age_days >= max_age_days:
            print(f"  ℹ️  OSM-Daten für {bundesland_code} sind {age_days} Tage alt → Download nötig")
            return True
        else:
            print(f"  ✓ OSM-Daten für {bundesland_code} sind {age_days} Tage alt → aktuell genug")
            return False
            
    except Exception as e:
        print(f"  ⚠️  Fehler beim Lesen der Metadata: {e} → Download wird durchgeführt")
        return True


def create_retry_session(total_retries=None, backoff_factor=None):
    """Erstellt eine requests-Session mit automatischen Retries für HTTP/Verbindungsfehler."""
    if total_retries is None:
        total_retries = PROCESSING_CONFIG.get("osm_download_retries", 5)
    if backoff_factor is None:
        backoff_factor = PROCESSING_CONFIG.get("osm_download_backoff_factor", 1.5)

    retry_strategy = Retry(
        total=total_retries,
        connect=total_retries,
        read=total_retries,
        status=total_retries,
        backoff_factor=backoff_factor,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False
    )

    adapter = HTTPAdapter(max_retries=retry_strategy)
    session = requests.Session()
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def is_temporary_network_error(error):
    """Klassifiziert typische temporäre Netzwerk-/DNS-Fehler."""
    error_text = str(error).lower()
    transient_markers = [
        "name resolution",
        "temporary failure in name resolution",
        "failed to resolve",
        "nodename nor servname provided",
        "connection reset",
        "timed out",
        "read timeout",
        "connect timeout",
        "newconnectionerror"
    ]
    return any(marker in error_text for marker in transient_markers)


def download_bundesland_pbf(bundesland_code, url, force_download=False, use_stale_fallback=None):
    """
    Download PBF file for a specific Bundesland.

    Returns:
        (file_path, is_fresh_download)
    """
    if use_stale_fallback is None:
        use_stale_fallback = PROCESSING_CONFIG.get("allow_stale_on_download_failure", True)

    folder_download = GEOFABRIK_CONFIG["download_folder"]
    os.makedirs(folder_download, exist_ok=True)
    
    filename = f"{bundesland_code}_{url.split('/')[-1]}"
    file_path = os.path.join(folder_download, filename)
    temp_path = f"{file_path}.tmp"

    # Bereits vorhanden und kein erzwungener Download nötig
    if os.path.exists(file_path) and not force_download:
        print(f"  ✓ Bereits vorhanden: {filename}")
        return file_path, False

    if os.path.exists(temp_path):
        os.remove(temp_path)

    max_attempts = PROCESSING_CONFIG.get("osm_download_manual_attempts", 3)
    base_delay_seconds = PROCESSING_CONFIG.get("osm_download_manual_base_delay", 5)
    session = create_retry_session()

    print(f"  📥 Lade herunter: {filename}")

    for attempt in range(1, max_attempts + 1):
        try:
            response = session.get(
                url,
                stream=True,
                timeout=(20, 300),
                allow_redirects=True
            )
            response.raise_for_status()

            total_size = int(response.headers.get('content-length', 0))

            with open(temp_path, "wb") as f, tqdm(
                total=total_size,
                unit='B',
                unit_scale=True,
                desc=f"    {bundesland_code}",
                leave=False
            ) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))

            os.replace(temp_path, file_path)
            print(f"  ✓ Download abgeschlossen: {filename}")
            return file_path, True

        except Exception as e:
            if os.path.exists(temp_path):
                os.remove(temp_path)

            transient = is_temporary_network_error(e)
            print(f"  ⚠️  Download-Versuch {attempt}/{max_attempts} für {bundesland_code} fehlgeschlagen: {e}")

            if attempt < max_attempts and transient:
                sleep_seconds = base_delay_seconds * (2 ** (attempt - 1)) + random.uniform(0, 1.5)
                print(f"     ↻ Warte {sleep_seconds:.1f}s und versuche erneut...")
                time.sleep(sleep_seconds)
                continue

            break

    print(f"  ❌ Fehler beim Download von {bundesland_code}: alle Versuche fehlgeschlagen")

    # Fallback: vorhandene Datei weiterverwenden (degradierter Modus)
    if use_stale_fallback and os.path.exists(file_path):
        print(f"  ⚠️  Fallback aktiv: Nutze vorhandene lokale Datei trotz fehlgeschlagenem Download")
        return file_path, False

    return None, False


def filter_highways_osmium(input_pbf, output_pbf, bundesland_code, force_filter=False):
    """Filter highways from PBF using Osmium."""
    if os.path.exists(output_pbf) and not force_filter:
        print(f"  ✓ Bereits gefiltert: {os.path.basename(output_pbf)}")
        return True
    
    if os.path.exists(output_pbf):
        print(f"  🔄 Überschreibe vorhandene gefilterte Datei")
        os.remove(output_pbf)
    
    print(f"  🔧 Filtere Highways für {bundesland_code}...")
    
    try:
        filter_command = [
            "osmium", "tags-filter",
            input_pbf,
            "w/highway",
            "-o", output_pbf
        ]
        subprocess.run(filter_command, check=True, capture_output=True)
        print(f"  ✓ Filterung abgeschlossen: {os.path.basename(output_pbf)}")
        return True
        
    except subprocess.CalledProcessError as e:
        print(f"  ❌ Fehler bei Osmium-Filterung für {bundesland_code}: {e}")
        if os.path.exists(output_pbf):
            os.remove(output_pbf)
        return False


def get_osm_timestamp(pbf_file):
    """Extract timestamp from PBF file using osmium."""
    try:
        result = subprocess.run([
            'osmium', 'fileinfo', pbf_file, '-g', 'header.option.timestamp'
        ], capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except subprocess.CalledProcessError:
        return None

In [5]:
# Hauptprozess: Download und Filterung für alle Bundesländer

folder_download = GEOFABRIK_CONFIG["download_folder"]
folder_processed = GEOFABRIK_CONFIG["processed_folder"]
os.makedirs(folder_download, exist_ok=True)
os.makedirs(folder_processed, exist_ok=True)

print(f"\n{'='*70}")
print(f"Download und Filterung OSM-Daten für {len(BUNDESLAND_URLS)} Bundesländer")
print(f"{'='*70}\n")

successful = []
failed = []
skipped = []
used_stale_fallback = []
timestamps = {}

for bundesland_code, url in BUNDESLAND_URLS.items():
    print(f"{'─'*70}")
    print(f"Bundesland: {bundesland_code}")
    print(f"{'─'*70}")
    
    try:
        # Prüfe ob Download nötig ist (4 Tage Grenze)
        needs_download = should_download_osm_data(bundesland_code)
        
        if not needs_download:
            # Daten sind aktuell genug → Skippen
            skipped.append(bundesland_code)
            
            # Timestamp aus Metadata laden für die Zusammenfassung
            try:
                with open("output/osm_metadata.json", "r") as f:
                    metadata = json.load(f)
                bundeslaender = metadata.get("bundeslaender", {})
                if bundesland_code in bundeslaender:
                    timestamps[bundesland_code] = bundeslaender[bundesland_code]
            except:
                pass
            
            print(f"  ⏩ {bundesland_code} übersprungen (Daten aktuell genug)\n")
            continue
        
        # 1. Download (nur wenn nötig)
        downloaded_pbf, is_fresh_download = download_bundesland_pbf(
            bundesland_code,
            url,
            force_download=needs_download
        )
        if not downloaded_pbf:
            failed.append(bundesland_code)
            continue
        
        # 2. Filter highways (nur wenn nötig)
        output_pbf = os.path.join(
            folder_processed,
            f"processed_highways_{bundesland_code}_latest.pbf"
        )

        # Bei fehlgeschlagenem Download mit Fallback:
        # vorhandenes Ergebnis bevorzugt behalten und nicht unnötig überschreiben
        if needs_download and not is_fresh_download:
            used_stale_fallback.append(bundesland_code)
            print("  ⚠️  Netzwerkproblem: Weiterarbeit mit vorhandenen lokalen Daten")

            if os.path.exists(output_pbf):
                timestamp = get_osm_timestamp(output_pbf)
                if timestamp:
                    timestamps[bundesland_code] = timestamp
                skipped.append(bundesland_code)
                print(f"  ⏩ {bundesland_code} ohne Re-Filterung weiterverwendet\n")
                continue
        
        success = filter_highways_osmium(
            downloaded_pbf,
            output_pbf,
            bundesland_code,
            force_filter=needs_download and is_fresh_download
        )
        
        if success:
            # 3. Extract timestamp
            timestamp = get_osm_timestamp(output_pbf)
            if timestamp:
                timestamps[bundesland_code] = timestamp
                print(f"  📅 OSM-Daten vom: {timestamp}")
            
            successful.append(bundesland_code)
            print(f"  ✅ {bundesland_code} erfolgreich verarbeitet!\n")
        else:
            failed.append(bundesland_code)
            
    except Exception as e:
        print(f"  ❌ Fehler bei {bundesland_code}: {e}\n")
        failed.append(bundesland_code)

# Zusammenfassung
print(f"\n{'='*70}")
print(f"ZUSAMMENFASSUNG")
print(f"{'='*70}")
print(f"✅ Erfolgreich: {len(successful)}/{len(BUNDESLAND_URLS)}")
if successful:
    print(f"   {', '.join(successful)}")

if skipped:
    print(f"\n⏩ Übersprungen (aktuell/weiterverwendet): {len(skipped)}/{len(BUNDESLAND_URLS)}")
    print(f"   {', '.join(skipped)}")

if used_stale_fallback:
    print(f"\n⚠️  Netzwerk-Fallback genutzt: {len(used_stale_fallback)}/{len(BUNDESLAND_URLS)}")
    print(f"   {', '.join(used_stale_fallback)}")

if failed:
    print(f"\n❌ Fehlgeschlagen: {len(failed)}/{len(BUNDESLAND_URLS)}")
    print(f"   {', '.join(failed)}")

# Speichere Metadata nur bei tatsächlich verarbeiteten Bundesländern
if successful and timestamps:
    oldest_date = min(timestamps.values())
    metadata = {
        "osm_data_from": oldest_date,
        "bundeslaender": timestamps,
        "processed_date": datetime.now().isoformat(),
        "download_fallback_used": used_stale_fallback
    }
    
    with open("output/osm_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    
    print(f"\n📅 Ältestes OSM-Datum: {oldest_date}")
    print(f"💾 Metadata gespeichert: output/osm_metadata.json")
elif used_stale_fallback:
    print("\nℹ️  Metadata nicht überschrieben, da keine frischen Downloads verarbeitet wurden.")

print(f"\n{'='*70}")
print(f"✅ FERTIG!")
print(f"{'='*70}")


Download und Filterung OSM-Daten für 16 Bundesländer

──────────────────────────────────────────────────────────────────────
Bundesland: DE-BW
──────────────────────────────────────────────────────────────────────
  ℹ️  OSM-Daten für DE-BW sind 6 Tage alt → Download nötig
  📥 Lade herunter: DE-BW_baden-wuerttemberg-latest.osm.pbf


  ✓ Download abgeschlossen: DE-BW_baden-wuerttemberg-latest.osm.pbf
  🔄 Überschreibe vorhandene gefilterte Datei
  🔧 Filtere Highways für DE-BW...


  ✓ Filterung abgeschlossen: processed_highways_DE-BW_latest.pbf
  📅 OSM-Daten vom: 2026-02-28T21:21:00Z
  ✅ DE-BW erfolgreich verarbeitet!

──────────────────────────────────────────────────────────────────────
Bundesland: DE-BY
──────────────────────────────────────────────────────────────────────
  ℹ️  OSM-Daten für DE-BY sind 6 Tage alt → Download nötig
  📥 Lade herunter: DE-BY_bayern-latest.osm.pbf


  ✓ Download abgeschlossen: DE-BY_bayern-latest.osm.pbf
  🔄 Überschreibe vorhandene gefilterte Datei
  🔧 Filtere Highways für DE-BY...


  ✓ Filterung abgeschlossen: processed_highways_DE-BY_latest.pbf
  📅 OSM-Daten vom: 2026-02-28T21:21:00Z
  ✅ DE-BY erfolgreich verarbeitet!

──────────────────────────────────────────────────────────────────────
Bundesland: DE-BE
──────────────────────────────────────────────────────────────────────
  ℹ️  OSM-Daten für DE-BE sind 6 Tage alt → Download nötig
  📥 Lade herunter: DE-BE_berlin-latest.osm.pbf


...

  ✓ Filterung abgeschlossen: processed_highways_DE-ST_latest.pbf
  📅 OSM-Daten vom: 2026-02-28T21:21:00Z
  ✅ DE-ST erfolgreich verarbeitet!

──────────────────────────────────────────────────────────────────────
Bundesland: DE-SH
──────────────────────────────────────────────────────────────────────
  ℹ️  OSM-Daten für DE-SH sind 6 Tage alt → Download nötig
  📥 Lade herunter: DE-SH_schleswig-holstein-latest.osm.pbf


  ✓ Download abgeschlossen: DE-SH_schleswig-holstein-latest.osm.pbf
  🔄 Überschreibe vorhandene gefilterte Datei
  🔧 Filtere Highways für DE-SH...


  ✓ Filterung abgeschlossen: processed_highways_DE-SH_latest.pbf
  📅 OSM-Daten vom: 2026-02-28T21:21:00Z
  ✅ DE-SH erfolgreich verarbeitet!

──────────────────────────────────────────────────────────────────────
Bundesland: DE-TH
──────────────────────────────────────────────────────────────────────
  ℹ️  OSM-Daten für DE-TH sind 6 Tage alt → Download nötig
  📥 Lade herunter: DE-TH_thueringen-latest.osm.pbf


  ✓ Download abgeschlossen: DE-TH_thueringen-latest.osm.pbf
  🔄 Überschreibe vorhandene gefilterte Datei
  🔧 Filtere Highways für DE-TH...


  ✓ Filterung abgeschlossen: processed_highways_DE-TH_latest.pbf
  📅 OSM-Daten vom: 2026-02-28T21:21:00Z
  ✅ DE-TH erfolgreich verarbeitet!


ZUSAMMENFASSUNG
✅ Erfolgreich: 16/16
   DE-BW, DE-BY, DE-BE, DE-BB, DE-HB, DE-HH, DE-HE, DE-MV, DE-NI, DE-NW, DE-RP, DE-SL, DE-SN, DE-ST, DE-SH, DE-TH

📅 Ältestes OSM-Datum: 2026-02-28T21:21:00Z
💾 Metadata gespeichert: output/osm_metadata.json

✅ FERTIG!


In [6]:
# Liste der verfügbaren gefilterten PBF-Dateien
import glob

pbf_files = glob.glob(f"{folder_processed}/processed_highways_DE-*_latest.pbf")

print(f"\n{'='*70}")
print(f"Verfügbare gefilterte PBF-Dateien: {len(pbf_files)}")
print(f"{'='*70}\n")

for pbf in sorted(pbf_files):
    size_mb = os.path.getsize(pbf) / (1024 * 1024)
    basename = os.path.basename(pbf)
    print(f"  {basename:50s} ({size_mb:6.1f} MB)")

# Zeige Metadata
if os.path.exists("output/osm_metadata.json"):
    with open("output/osm_metadata.json", "r") as f:
        metadata = json.load(f)
    
    print(f"\n📅 OSM-Daten vom: {metadata.get('osm_data_from', 'Unknown')}")
    print(f"🕒 Verarbeitet am: {metadata.get('processed_date', 'Unknown')}")
else:
    print("\n⚠️  Keine Metadata gefunden")


Verfügbare gefilterte PBF-Dateien: 16

  processed_highways_DE-BB_latest.pbf                (  66.7 MB)
  processed_highways_DE-BE_latest.pbf                (  24.2 MB)
  processed_highways_DE-BW_latest.pbf                ( 169.8 MB)
  processed_highways_DE-BY_latest.pbf                ( 227.3 MB)
  processed_highways_DE-HB_latest.pbf                (   4.3 MB)
  processed_highways_DE-HE_latest.pbf                (  86.1 MB)
  processed_highways_DE-HH_latest.pbf                (  11.8 MB)
  processed_highways_DE-MV_latest.pbf                (  24.6 MB)
  processed_highways_DE-NI_latest.pbf                ( 107.5 MB)
  processed_highways_DE-NW_latest.pbf                ( 185.5 MB)
  processed_highways_DE-RP_latest.pbf                (  72.5 MB)
  processed_highways_DE-SH_latest.pbf                (  33.4 MB)
  processed_highways_DE-SL_latest.pbf                (  11.7 MB)
  processed_highways_DE-SN_latest.pbf                (  64.4 MB)
  processed_highways_DE-ST_latest.pbf             